In [ ]:
import chess


def generate_realistic_move_vocabulary():
    """
    Generate all moves that can exist in a real chess game.
    Uses python-chess to ensure each move could be legal in some position.
    Returns a sorted list of UCI strings (e.g. 'e2e4', 'e7e8q', 'g1f3', ...).
    """
    realistic_moves = set()

    # Iterate over all squares
    for square in chess.SQUARES:
        for piece_type in [2, 5]:  # pawn=1, knight=2, ..., king=6
            for color in [chess.WHITE, chess.BLACK]:
                # Create an empty board
                b = chess.Board.empty()

                # Place only one piece of the given type/color
                b.set_piece_at(square, chess.Piece(piece_type, color))

                # Generate all legal moves for this configuration
                for move in b.legal_moves:
                    realistic_moves.add(move.uci())

    # Also include all promotion variants
    for file in "abcdefgh":
        for promo in "qrbn":
            realistic_moves.add(f"{file}7{file}8{promo}")
            realistic_moves.add(f"{file}2{file}1{promo}")

    file_val = "abcdefgh"
    for i, file in enumerate(file_val):
        for promo in "qrbn":
            if i > 0:
                realistic_moves.add(f"{file}7{file_val[i - 1]}8{promo}")
                realistic_moves.add(f"{file}2{file_val[i - 1]}1{promo}")
            if i < len(file_val) - 1:
                realistic_moves.add(f"{file}7{file_val[i + 1]}8{promo}")
                realistic_moves.add(f"{file}2{file_val[i + 1]}1{promo}")

    # Sort for consistency
    vocab = sorted(realistic_moves)
    vocab_dict = {move: idx for idx, move in enumerate(vocab)}
    return vocab_dict

In [ ]:
moves = generate_realistic_move_vocabulary()
print(len(moves))
print(moves)

In [ ]:
import json


def save_move_vocab(filename="uci_vocab.json"):
    vocab = generate_realistic_move_vocabulary()
    with open(filename, "w") as f:
        json.dump(vocab, f, indent=4)


save_move_vocab()

In [ ]:
ascii_board_vocab = {
    "P": 1,  # White Pawn
    "N": 2,  # White Knight
    "B": 3,  # White Bishop
    "R": 4,  # White Rook
    "Q": 5,  # White Queen
    "K": 6,  # White King
    "p": 7,  # Black Pawn
    "n": 8,  # Black Knight
    "b": 9,  # Black Bishop
    "r": 10,  # Black Rook
    "q": 11,  # Black Queen
    "k": 12,  # Black King
    ".": 0,  # Empty square
}


def save_ascii_board_vocab(filename="ascii_board_vocab.json"):
    with open(filename, "w") as f:
        json.dump(ascii_board_vocab, f, indent=4)


save_ascii_board_vocab()

In [ ]:
import chess.svg
from PIL import Image
from io import BytesIO
import cairosvg


def generate_vocab_building_gif(filename="vocab_building_process.gif", size=400, duration=300, max_frames=200):
    """
    Generate an animated GIF showing how we build the chess move vocabulary.

    Focuses on:
    1. White Queen on all squares
    2. White Knight on all squares
    3. Pawn promotions (both White and Black) - showing each promotion piece

    Args:
        filename: Output filename for the GIF
        size: Size of each board frame in pixels
        duration: Duration of each frame in milliseconds
        max_frames: Maximum number of frames to include (to keep file size reasonable)

    Returns:
        Path to the saved GIF file
    """
    frames = []
    frame_count = 0
    moves_found = 0
    unique_moves = set()  # Track unique move strings

    piece_names = {2: "Knight", 5: "Queen"}

    print(f"Generating vocabulary building GIF...")
    print(f"Will show: White Queen, White Knight, and Pawn Promotions")

    # Part 1: White Queen on all squares
    print("\n1. Processing White Queen...")
    for square in chess.SQUARES:
        square_name = chess.square_name(square)

        # Create an empty board with White Queen
        board = chess.Board.empty()
        board.set_piece_at(square, chess.Piece(5, chess.WHITE))  # 5 = Queen

        # Count legal moves from this position
        legal_moves = list(board.legal_moves)
        previous_count = len(unique_moves)
        for move in legal_moves:
            unique_moves.add(move.uci())
        new_moves_added = len(unique_moves) - previous_count
        moves_found += len(legal_moves)

        # Generate SVG with arrows showing legal moves
        arrows = [(move.from_square, move.to_square) for move in legal_moves]
        svg_data = chess.svg.board(board, size=size, arrows=arrows, colors={"arrow": "green"})

        # Add text annotation
        svg_lines = svg_data.split("\n")
        text_annotation = f'''
  <text x="10" y="30" font-size="20" font-weight="bold" fill="black" 
        stroke="white" stroke-width="0.5">White Queen</text>
  <text x="10" y="55" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Square: {square_name}</text>
  <text x="10" y="80" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Legal moves: {len(legal_moves)}</text>
  <text x="10" y="105" font-size="14" fill="gray" 
        stroke="white" stroke-width="0.5">Unique moves so far: {len(unique_moves)}</text>
  <text x="{size - 10}" y="30" font-size="24" font-weight="bold" fill="darkgreen" 
        stroke="white" stroke-width="0.5" text-anchor="end">+{new_moves_added}</text>
'''
        svg_data = "\n".join(svg_lines[:-1]) + text_annotation + "\n" + svg_lines[-1]

        # Convert SVG to PNG
        png_data = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
        img = Image.open(BytesIO(png_data))
        frames.append(img)
        frame_count += 1

    # Part 2: White Knight on all squares
    print("2. Processing White Knight...")
    for square in chess.SQUARES:
        square_name = chess.square_name(square)

        # Create an empty board with White Knight
        board = chess.Board.empty()
        board.set_piece_at(square, chess.Piece(2, chess.WHITE))  # 2 = Knight

        # Count legal moves from this position
        legal_moves = list(board.legal_moves)
        previous_count = len(unique_moves)
        for move in legal_moves:
            unique_moves.add(move.uci())
        new_moves_added = len(unique_moves) - previous_count
        moves_found += len(legal_moves)

        # Generate SVG with arrows showing legal moves
        arrows = [(move.from_square, move.to_square) for move in legal_moves]
        svg_data = chess.svg.board(board, size=size, arrows=arrows, colors={"arrow": "blue"})

        # Add text annotation
        svg_lines = svg_data.split("\n")
        text_annotation = f'''
  <text x="10" y="30" font-size="20" font-weight="bold" fill="black" 
        stroke="white" stroke-width="0.5">White Knight</text>
  <text x="10" y="55" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Square: {square_name}</text>
  <text x="10" y="80" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Legal moves: {len(legal_moves)}</text>
  <text x="10" y="105" font-size="14" fill="gray" 
        stroke="white" stroke-width="0.5">Unique moves so far: {len(unique_moves)}</text>
  <text x="{size - 10}" y="30" font-size="24" font-weight="bold" fill="darkblue" 
        stroke="white" stroke-width="0.5" text-anchor="end">+{new_moves_added}</text>
'''
        svg_data = "\n".join(svg_lines[:-1]) + text_annotation + "\n" + svg_lines[-1]

        # Convert SVG to PNG
        png_data = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
        img = Image.open(BytesIO(png_data))
        frames.append(img)
        frame_count += 1

    # Part 3: Pawn Promotions (both colors) - showing each promotion piece type
    print("3. Processing Pawn Promotions...")

    promotion_pieces = {
        chess.QUEEN: ("Queen", "♕"),
        chess.ROOK: ("Rook", "♖"),
        chess.BISHOP: ("Bishop", "♗"),
        chess.KNIGHT: ("Knight", "♘"),
    }

    # White pawn promotions - for each file and each promotion piece
    for file_idx in range(8):  # a-h files
        file_letter = chr(ord("a") + file_idx)

        for promo_piece, (piece_name, piece_symbol) in promotion_pieces.items():
            # Create board with white pawn on 7th rank
            board = chess.Board.empty()
            from_square = chess.parse_square(f"{file_letter}7")
            to_square = chess.parse_square(f"{file_letter}8")
            board.set_piece_at(from_square, chess.Piece(1, chess.WHITE))  # White Pawn

            # Get this specific promotion move
            move_uci = f"{file_letter}7{file_letter}8{chess.piece_symbol(promo_piece).lower()}"
            previous_count = len(unique_moves)
            unique_moves.add(move_uci)
            new_moves_added = len(unique_moves) - previous_count
            moves_found += 1

            # Create the resulting board (after promotion)
            result_board = chess.Board.empty()
            result_board.set_piece_at(to_square, chess.Piece(promo_piece, chess.WHITE))

            # Show both before and after
            svg_data = chess.svg.board(
                board,
                size=size,
                arrows=[(from_square, to_square)],
                colors={"arrow": "gold"},
            )

            # Add text annotation
            svg_lines = svg_data.split("\n")
            text_annotation = f'''
  <text x="10" y="30" font-size="20" font-weight="bold" fill="black" 
        stroke="white" stroke-width="0.5">White Pawn → {piece_name} {piece_symbol}</text>
  <text x="10" y="55" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Move: {file_letter}7{file_letter}8{chess.piece_symbol(promo_piece).lower()}</text>
  <text x="10" y="80" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">File: {file_letter}</text>
  <text x="10" y="105" font-size="14" fill="gray" 
        stroke="white" stroke-width="0.5">Unique moves so far: {len(unique_moves)}</text>
  <text x="{size - 10}" y="30" font-size="24" font-weight="bold" fill="goldenrod" 
        stroke="white" stroke-width="0.5" text-anchor="end">+{new_moves_added}</text>
'''
            svg_data = "\n".join(svg_lines[:-1]) + text_annotation + "\n" + svg_lines[-1]

            png_data = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
            img = Image.open(BytesIO(png_data))
            frames.append(img)
            frame_count += 1

    # Black pawn promotions - for each file and each promotion piece
    for file_idx in range(8):  # a-h files
        file_letter = chr(ord("a") + file_idx)

        for promo_piece, (piece_name, piece_symbol) in promotion_pieces.items():
            # Create board with black pawn on 2nd rank
            board = chess.Board.empty()
            from_square = chess.parse_square(f"{file_letter}2")
            to_square = chess.parse_square(f"{file_letter}1")
            board.set_piece_at(from_square, chess.Piece(1, chess.BLACK))  # Black Pawn
            board.turn = chess.BLACK  # Set turn to black

            # Get this specific promotion move
            move_uci = f"{file_letter}2{file_letter}1{chess.piece_symbol(promo_piece).lower()}"
            previous_count = len(unique_moves)
            unique_moves.add(move_uci)
            new_moves_added = len(unique_moves) - previous_count
            moves_found += 1

            # Show the board
            svg_data = chess.svg.board(
                board,
                size=size,
                arrows=[(from_square, to_square)],
                colors={"arrow": "orange"},
                orientation=chess.BLACK,
            )

            # Add text annotation
            svg_lines = svg_data.split("\n")
            text_annotation = f'''
  <text x="10" y="30" font-size="20" font-weight="bold" fill="black" 
        stroke="white" stroke-width="0.5">Black Pawn → {piece_name} {piece_symbol}</text>
  <text x="10" y="55" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">Move: {file_letter}2{file_letter}1{chess.piece_symbol(promo_piece).lower()}</text>
  <text x="10" y="80" font-size="16" fill="black" 
        stroke="white" stroke-width="0.5">File: {file_letter}</text>
  <text x="10" y="105" font-size="14" fill="gray" 
        stroke="white" stroke-width="0.5">Unique moves so far: {len(unique_moves)}</text>
  <text x="{size - 10}" y="30" font-size="24" font-weight="bold" fill="darkorange" 
        stroke="white" stroke-width="0.5" text-anchor="end">+{new_moves_added}</text>
'''
            svg_data = "\n".join(svg_lines[:-1]) + text_annotation + "\n" + svg_lines[-1]

            png_data = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
            img = Image.open(BytesIO(png_data))
            frames.append(img)
            frame_count += 1

    # Add a final summary frame
    board = chess.Board.empty()
    svg_data = chess.svg.board(board, size=size)
    svg_lines = svg_data.split("\n")
    summary_text = f'''
  <text x="{size // 2}" y="{size // 2 - 100}" font-size="28" font-weight="bold" 
        fill="darkgreen" text-anchor="middle">Vocabulary Generation Complete!</text>
  <text x="{size // 2}" y="{size // 2 - 60}" font-size="22" fill="black" 
        text-anchor="middle">Total UNIQUE moves: {len(unique_moves)}</text>
  <text x="{size // 2}" y="{size // 2 - 30}" font-size="18" fill="gray" 
        text-anchor="middle">Total move instances: {moves_found}</text>
  <text x="{size // 2}" y="{size // 2 + 10}" font-size="16" fill="gray" 
        text-anchor="middle">White Queen: 64 squares</text>
  <text x="{size // 2}" y="{size // 2 + 35}" font-size="16" fill="gray" 
        text-anchor="middle">White Knight: 64 squares</text>
  <text x="{size // 2}" y="{size // 2 + 60}" font-size="16" fill="gray" 
        text-anchor="middle">White Promotions: 32 (8 files × 4 pieces)</text>
  <text x="{size // 2}" y="{size // 2 + 85}" font-size="16" fill="gray" 
        text-anchor="middle">Black Promotions: 32 (8 files × 4 pieces)</text>
  <text x="{size // 2}" y="{size // 2 + 110}" font-size="14" fill="darkgray" 
        text-anchor="middle">Total frames: {frame_count}</text>
'''
    svg_data = "\n".join(svg_lines[:-1]) + summary_text + "\n" + svg_lines[-1]
    png_data = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
    img = Image.open(BytesIO(png_data))
    frames.append(img)

    # Save as animated GIF
    print(f"\nSaving GIF with {len(frames)} frames...")
    frames[0].save(
        filename,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0,
        optimize=False,
    )

    print(f"✅ GIF saved to: {filename}")
    print(f"   Total frames: {len(frames)}")
    print(f"   Total UNIQUE moves found: {len(unique_moves)}")
    print(f"   Total move instances: {moves_found}")
    print(f"   Breakdown:")
    print(f"     - White Queen: 64 frames")
    print(f"     - White Knight: 64 frames")
    print(f"     - White Pawn Promotions: 32 frames (8 files × 4 pieces)")
    print(f"     - Black Pawn Promotions: 32 frames (8 files × 4 pieces)")
    print(f"     - Summary: 1 frame")

    return filename

In [ ]:
# Generate the GIF showing vocabulary building for:
# - White Queen (all 64 squares)
# - White Knight (all 64 squares)
# - White Pawn Promotions (8 files × 4 pieces = 32 frames)
# - Black Pawn Promotions (8 files × 4 pieces = 32 frames)
# Total: 193 frames (64 + 64 + 32 + 32 + 1 summary)

gif_path = generate_vocab_building_gif(
    filename="chess_vocab_queen_knight_promotions.gif",
    size=400,
    duration=150,  # 150ms per frame
    max_frames=200,
)

print(f"\n✨ GIF created! You can view it at: {gif_path}")